# fetch-missing-logos

**Purpose:** Discover logos for partner records in Airtable that have no `Organization logo (BW)` attachment, then optionally write the found logos back to Airtable.

This is a **one-off / maintenance utility**, not part of the regular data pipeline.

## When to run
- A batch of new partners was added to Airtable and most lack a BW logo.
- After running, review the downloaded files in `public/logos/partners/white/`
  before committing Cell 9 (Airtable write-back).

## Logo sources (tried in order)
1. **Clearbit Logo API** — `https://logo.clearbit.com/{domain}` — high-quality PNGs/SVGs
2. **Google S2 favicon** — `https://www.google.com/s2/favicons?domain={domain}&sz=256` — fallback

## Workflow
1. Run cells 1–7 to fetch, filter, and download logos locally.
2. Inspect the results summary (cell 8) and the downloaded files.
3. **Only** run cell 9 (Airtable write-back) once you are satisfied with the results.
4. Re-run `01-fetch_partners` to regenerate `partners.json` with the new logos.

> **Note:** Logos are saved to `public/logos/partners/white/` — the same directory
> that `01-fetch_partners` writes to. Files already present are skipped.

In [ ]:
# ── Cell 1 · Imports ──────────────────────────────────────────────────────────
import os
import re
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv
from pyairtable import Api

load_dotenv()
print("imports OK")

In [ ]:
# ── Cell 2 · Constants ────────────────────────────────────────────────────────
AIRTABLE_BASE_ID = "appIYFN5sAJzK1bPg"
PARTNER_TABLE_ID = "tbl2FMZOARI7I66fq"

# Exact Airtable field name for the BW logo attachment — used for write-back.
LOGO_FIELD_NAME = "Organization logo (BW)"

# Logos are written here so 01-fetch_partners picks them up on the next run.
LOGOS_DIR = Path("public") / "logos" / "partners" / "white"

CLEARBIT_TMPL = "https://logo.clearbit.com/{domain}"
GOOGLE_S2_TMPL = "https://www.google.com/s2/favicons?domain={domain}&sz=256"

HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; logo-fetcher/1.0)"}
TIMEOUT = 15

# Google S2 returns this exact byte-count for its generic placeholder — skip it.
GOOGLE_FALLBACK_SIZE_BYTES = 1_114

LOGOS_DIR.mkdir(parents=True, exist_ok=True)
print(f"logos dir: {LOGOS_DIR.resolve()}")

In [ ]:
# ── Cell 3 · Load partners from Airtable ─────────────────────────────────────
# cell_format="json" (pyairtable default): attachment fields come back as
# lists of dicts [{id, url, filename, type, size, ...}] or are absent entirely.
# A missing or empty-list attachment field means "no logo".

api = Api(os.environ["AIRTABLE_API_KEY"])
table = api.table(AIRTABLE_BASE_ID, PARTNER_TABLE_ID)
raw_records = table.all()

rows = []
for rec in raw_records:
    row = {"_record_id": rec["id"]}
    row.update(rec["fields"])
    rows.append(row)

df_raw = pd.DataFrame(rows)
print(f"fetched {len(df_raw)} records")

In [ ]:
# ── Cell 4 · Rename, select, and filter to missing-logo rows ──────────────────
rename_mapping = {
    "Organization name": "org_full_name",
    "Short name": "org_short_name",
    "Website": "org_url",
    "Organization logo (BW)": "org_logo_white",
}
df = df_raw.rename(columns=rename_mapping)

keep = ["_record_id", "org_short_name", "org_full_name", "org_url", "org_logo_white"]
df = df[[c for c in keep if c in df.columns]].copy()


# In JSON mode, absent attachment field → NaN; empty list → []
# Both mean "no logo uploaded in Airtable".
def _has_logo(val) -> bool:
    return isinstance(val, list) and len(val) > 0


df_missing = df[
    ~df.get("org_logo_white", pd.Series(dtype=object)).apply(_has_logo)
].copy()
df_missing = df_missing.reset_index(drop=True)

print(f"total records  : {len(df)}")
print(f"already have logo: {len(df) - len(df_missing)}")
print(f"missing logo   : {len(df_missing)}")
df_missing[["org_short_name", "org_full_name", "org_url"]]

In [ ]:
# ── Cell 5 · Domain normaliser ────────────────────────────────────────────────
def normalise_domain(website) -> str | None:
    """Strip protocol, www, trailing slash and path → bare hostname or None."""
    url = str(website).strip() if website else ""
    if not url or url == "nan":
        return None
    if not re.match(r"https?://", url, re.I):
        url = "https://" + url
    host = urlparse(url).hostname or ""
    host = re.sub(r"^www\.", "", host).lower().strip(".")
    return host or None


df_missing["domain"] = df_missing["org_url"].apply(normalise_domain)
print(f"usable domains : {df_missing['domain'].notna().sum()} / {len(df_missing)}")
df_missing[["org_short_name", "org_url", "domain"]]

In [ ]:
# ── Cell 6 · Logo fetch helpers ───────────────────────────────────────────────
VALID_CONTENT_TYPES = {
    "image/png",
    "image/jpeg",
    "image/jpg",
    "image/svg+xml",
    "image/webp",
}


def _ext_from_content_type(ct: str) -> str:
    return {
        "image/png": ".png",
        "image/jpeg": ".jpg",
        "image/jpg": ".jpg",
        "image/svg+xml": ".svg",
        "image/webp": ".webp",
    }.get(ct.split(";")[0].strip().lower(), ".png")


def _to_slug(name: str) -> str:
    return (
        str(name)
        .lower()
        .replace(" ", "-")
        .replace("/", "-")
        .replace("_", "-")
        .replace("&", "")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
    )


def fetch_logo(url: str, skip_if_bytes: int | None = None) -> requests.Response | None:
    """GET url; return Response if it looks like a valid image, else None."""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        resp.raise_for_status()
    except requests.RequestException:
        return None
    ct = resp.headers.get("Content-Type", "")
    if not any(ct.startswith(v) for v in VALID_CONTENT_TYPES):
        return None
    if skip_if_bytes is not None and len(resp.content) == skip_if_bytes:
        return None
    return resp


def fetch_logo_for_domain(
    domain: str,
) -> tuple[requests.Response, str] | tuple[None, None]:
    """Try Clearbit then Google S2. Returns (response, source_label) or (None, None)."""
    resp = fetch_logo(CLEARBIT_TMPL.format(domain=domain))
    if resp:
        return resp, "clearbit"
    resp = fetch_logo(
        GOOGLE_S2_TMPL.format(domain=domain), skip_if_bytes=GOOGLE_FALLBACK_SIZE_BYTES
    )
    if resp:
        return resp, "google_s2"
    return None, None


print("helpers defined")

In [ ]:
# ── Cell 7 · Download loop ────────────────────────────────────────────────────
# Saves files to LOGOS_DIR (public/logos/partners/white/).
# Skips partners with no website domain and files already present on disk.
#
# Statuses written to `results`:
#   missing_website  — no usable domain
#   already_local    — file already exists, skipped
#   logo_not_found   — both sources returned nothing
#   downloaded       — saved successfully


def _coerce_str(val) -> str:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()


results: list[dict] = []

for _, row in df_missing.iterrows():
    rec_id = row["_record_id"]
    name = (
        _coerce_str(row.get("org_short_name"))
        or _coerce_str(row.get("org_full_name"))
        or rec_id
    )
    domain = row.get("domain")
    base = {
        "record_id": rec_id,
        "org_short_name": name,
        "source": None,
        "local_path": None,
    }

    if not domain:
        results.append({**base, "status": "missing_website"})
        print(f"  ⊘  {name:<35}  no website")
        continue

    slug = _to_slug(name)
    existing = list(LOGOS_DIR.glob(f"{slug}.*"))
    if existing:
        results.append(
            {**base, "status": "already_local", "local_path": str(existing[0])}
        )
        print(f"  ↩  {name:<35}  already local → {existing[0].name}")
        continue

    try:
        resp, source = fetch_logo_for_domain(domain)
    except Exception as exc:
        results.append({**base, "status": "error", "source": str(exc)})
        print(f"  ✗  {name:<35}  exception: {exc}")
        continue

    if resp is None:
        results.append({**base, "status": "logo_not_found"})
        print(f"  ✗  {name:<35}  not found (domain: {domain})")
        continue

    ext = _ext_from_content_type(resp.headers.get("Content-Type", ""))
    filepath = LOGOS_DIR / f"{slug}{ext}"
    filepath.write_bytes(resp.content)

    results.append(
        {**base, "status": "downloaded", "local_path": str(filepath), "source": source}
    )
    print(f"  ✓  {name:<35}  {source:<12}  → {filepath.name}")
    time.sleep(0.3)

print("\nDone.")

In [ ]:
# ── Cell 8 · Results summary ──────────────────────────────────────────────────
df_results = pd.DataFrame(results)

print("=== outcome ===")
print(df_results.groupby("status").size().rename("count").to_string())

needs_review = df_results[
    df_results["status"].isin(["logo_not_found", "missing_website", "error"])
]
if not needs_review.empty:
    print("\n--- needs manual review ---")
    print(needs_review[["org_short_name", "status"]].to_string(index=False))

df_results

In [ ]:
# ── Cell 9 · Write back to Airtable ──────────────────────────────────────────
#
# !! FINAL STEP — only run after reviewing the downloaded files in LOGOS_DIR !!
#
# Uploads each downloaded logo to the partner's Airtable record.
# Only processes records with status == 'downloaded'; skips everything else.
# After this cell, re-run 01-fetch_partners to regenerate partners.json.

to_upload = df_results[df_results["status"] == "downloaded"].copy()
to_upload = to_upload[to_upload["local_path"].notna()]
print(f"Uploading {len(to_upload)} logo(s) to Airtable...\n")

upload_log: list[dict] = []

for _, row in to_upload.iterrows():
    rec_id = row["record_id"]
    name = row["org_short_name"]
    fpath = Path(row["local_path"])

    if not fpath.exists():
        upload_log.append({"org_short_name": name, "status": "file_missing"})
        print(f"  ✗  {name:<35}  file not found: {fpath}")
        continue

    try:
        table.upload_attachment(
            record_id=rec_id,
            field_name=LOGO_FIELD_NAME,
            filename=fpath.name,
            content=fpath.read_bytes(),
        )
        upload_log.append(
            {"org_short_name": name, "status": "uploaded", "file": fpath.name}
        )
        print(f"  ✓  {name:<35}  → {fpath.name}")
    except Exception as exc:
        upload_log.append(
            {"org_short_name": name, "status": "failed", "error": str(exc)}
        )
        print(f"  ✗  {name:<35}  upload failed: {exc}")

    time.sleep(0.2)

print("\n=== upload summary ===")
print(pd.DataFrame(upload_log).groupby("status").size().to_string())
print("\nNext step: run  python -m python.partners.01-fetch_partners")